# 12_04 On your own: a grounded support assistant for Kittiwake

Kittiwake wants an assistant that answers customers' questions from its service notices, on its own
hardware, and says so when the notices do not cover a question. Build `answer(question)`, run it on the
fixed question set, and save the results. The brief is on the lab page.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-12-large-and-small-language-models", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json, time
import slm
from nlpcheck import check_on_your_own, QUESTION_SET
N = slm.notices()
tok, model = slm.load()
for q, _ in QUESTION_SET:
    print(q)

## Look before you build

Retrieval scores for every question, both ways. Which questions have a weak best match? Which notices come
back for the questions the notices cannot answer?

In [ ]:
for q, _ in QUESTION_SET:
    print(q)
    print("   BM25:      ", slm.bm25_search(q, 2))
    print("   embeddings:", slm.embed_search(q, 2))

## Your assistant

`answer()` must return a dict with `"answer"`, and whenever it asks the model, the `"messages"` it sent and
the `"max_new_tokens"` it allowed, so the checkpoint can replay one. Keep `max_new_tokens` small (24 to
32): the whole set should run in a couple of minutes. Decline with a sentence containing "I don't know".

In [ ]:
MAX_NEW = 24

def answer(question):
    # YOUR CODE HERE: retrieve notices, decide whether to answer at all, build the messages, generate
    passages, messages = [], None
    reply = "I don't know."
    return {"answer": reply, "passages": passages, "messages": messages, "max_new_tokens": MAX_NEW}

## Run the set

Eight questions, one generation each. It takes about two minutes; each line prints as it finishes.

In [ ]:
results = slm.run_question_set(answer)

In [ ]:
check_on_your_own()

Before you sign off, read every answer, not just the score: an answer can contain the right number and
still say something the notice does not. Which of your design choices mattered most?

*Your notes:* 